In [ ]:
!pip -q install transformers sentencepiece evaluate rouge_score

In [ ]:
import time
import torch
import pandas as pd
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
article = """Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning
models to read X-ray and MRI images, and these systems can detect diseases such as cancer and pneumonia in
very early stages. Hospitals also use AI to predict which patients are likely to need intensive care, so that
beds and staff can be arranged in advance. Chatbots answer simple questions of patients, book appointments and
send reminders for medicines, which reduces the workload of nurses. However, experts warn that these systems
must be trained on diverse and high quality data, otherwise they may give wrong results for some groups of
people. Privacy of patient records is another big concern, and governments are preparing strict rules for the
use of medical data. Most researchers agree that AI will not replace doctors but will help them to take faster
and better decisions."""

reference = ("AI helps hospitals detect diseases early, predict patient needs and reduce staff workload, "
             "but needs good data and privacy rules, and will assist rather than replace doctors.")
print("Article words:", len(article.split()))

In [ ]:
def summarize(model_name, text, prefix="", **gen_kwargs):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()
    enc = tok(prefix + text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    start = time.time()
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    return tok.decode(out[0], skip_special_tokens=True), round(time.time() - start, 2)

gen = dict(num_beams=4, max_length=70, min_length=25, early_stopping=True, no_repeat_ngram_size=3)

bart_sum, bart_t = summarize("facebook/bart-large-cnn", article, **gen)
print(f"BART summary ({bart_t}s):\n{bart_sum}\n")

In [ ]:
t5_sum, t5_t = summarize("t5-small", article, prefix="summarize: ", **gen)
print(f"T5-small summary ({t5_t}s):\n{t5_sum}")

In [ ]:
# ROUGE evaluation against the reference summary
rouge = evaluate.load("rouge")
rows = []
for name, s in [("BART", bart_sum), ("T5-small", t5_sum)]:
    r = rouge.compute(predictions=[s], references=[reference])
    rows.append((name, len(s.split()), round(r["rouge1"], 3), round(r["rouge2"], 3), round(r["rougeL"], 3)))
display(pd.DataFrame(rows, columns=["model", "summary words", "ROUGE-1", "ROUGE-2", "ROUGE-L"]))
print("Compression: article", len(article.split()), "words ->", len(bart_sum.split()), "words (BART)")

In [ ]:
# Effect of summary length (BART)
for mx in [30, 60, 100]:
    s, _ = summarize("facebook/bart-large-cnn", article, num_beams=4, max_length=mx, min_length=10, early_stopping=True)
    print(f"max_length={mx:3d} -> {s}\n")